# Quantum Algorithms: Grover Search & Teleportation (Qiskit)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/09_Other_Experiments/quantum/quantum_grover_teleportation.ipynb)

Two landmark protocols beyond basic circuits: **Grover's algorithm** (quadratic speedup for unstructured search) and **quantum teleportation** (moving a qubit state using entanglement + 2 classical bits).

Runs on the free AerSimulator - no quantum hardware needed.

In [ ]:
!pip install -q qiskit qiskit-aer matplotlib pylatexenc

## 1. Grover: find the marked item among 4

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

n = 2                                   # search space = 2^n = 4 items
qc = QuantumCircuit(n, n)
qc.h(range(n))                          # superposition over all items

oracle = QuantumCircuit(n, name="oracle")   # mark |11>
oracle.cz(0, 1)

diffuser = QuantumCircuit(n, name="diffuser")   # amplitude amplification
diffuser.h(range(n)); diffuser.x(range(n))
diffuser.cz(0, 1); diffuser.x(range(n)); diffuser.h(range(n))

for _ in range(1):                      # optimal iterations ~ pi/4 * sqrt(N) = 1
    qc.compose(oracle, inplace=True)
    qc.compose(diffuser, inplace=True)
qc.measure(range(n), range(n))
sim = AerSimulator()
counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
plot_histogram(counts); plt.show()

`11` wins ~100% of shots after just ONE Grover iteration - classically you would need up to 3 queries; at scale N=10^6 Grover needs ~1000 vs 500k classical.

## 2. Circuit anatomy

In [ ]:
grover_only = QuantumCircuit(n, n)
grover_only.compose(oracle, inplace=True)
grover_only.compose(diffuser, inplace=True)
print(grover_only.decompose().draw(fold=120))

## 3. Quantum teleportation

In [ ]:
tele = QuantumCircuit(3, 3)

# state to teleport on qubit 0
tele.ry(1.2345, 0)

# Bell pair between qubits 1 (sender) and 2 (receiver)
tele.h(1); tele.cx(1, 2); tele.barrier()

# sender's Bell measurement basis
tele.cx(0, 1); tele.h(0); tele.barrier()
tele.measure([0, 1], [0, 1]); tele.barrier()

# receiver applies corrections from the 2 classical bits
tele.cx(1, 2); tele.cz(0, 2)
tele.measure(2, 2)
print(tele.draw(fold=120))

In [ ]:
counts_t = sim.run(transpile(tele, sim), shots=2048).result().get_counts()
recv = {k[-1]: v for k, v in counts_t.items()}      # last bit = received qubit
plot_histogram(recv, title="received qubit collapses to |1> (ry angle was positive)")
plt.show()

## Key takeaways
- Teleportation moves a STATE (destroying the original) - no faster-than-light info; classical bits required.
- Grover is provably optimal for unstructured search and underpins quantum attack analysis of symmetric crypto.
- Real hardware adds noise: swap `AerSimulator()` for `IBMQ` backends later - code identical.